# Week 3 — Model Evaluation & Tuning

This section revisits the Week 2 Titanic classification model to evaluate it 
beyond accuracy, and improves it using hyperparameter tuning.

### 1. Setup & Imports

In [19]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, accuracy_score

### 2. Load Cleaned Dataset

In [20]:
# Load the cleaned Titanic dataset from Week 1
df = pd.read_csv("../data/titanic_cleaned.csv")

### 3. Encode Categorical Columns

In [21]:
df_encoded = pd.get_dummies(df, columns=['Sex', 'Embarked'], drop_first=True)

### 4. Train Baseline Model (Logistic Regression)

In [22]:
# Features and target
X = df_encoded.drop(['Survived', 'PassengerId', 'Name', 'Ticket'], axis=1)
y = df_encoded['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

print("Model trained successfully")
print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

Model trained successfully
Training set: (712, 9)
Test set: (179, 9)


### 5. Classification Report (Precision, Recall, F1-score)

In [23]:
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Not Survived', 'Survived']))

Accuracy: 0.8212290502793296

Classification Report:
              precision    recall  f1-score   support

Not Survived       0.83      0.87      0.85       105
    Survived       0.80      0.76      0.78        74

    accuracy                           0.82       179
   macro avg       0.82      0.81      0.81       179
weighted avg       0.82      0.82      0.82       179



### 6. Why Accuracy Alone Can Be Misleading

- Our test set has **105 "Not Survived"** vs only **74 "Survived"** — a moderate class imbalance.
- If a model just predicted "Not Survived" for everyone, it would still get **~59% accuracy** (105/179) without learning anything meaningful.
- **Precision** tells us: of everyone the model predicted as "Survived," how many actually did (80%).
- **Recall** tells us: of everyone who actually survived, how many did the model catch (76%).
- The **F1-score** balances both — useful when false positives and false negatives both matter.
- Here, "Survived" has slightly lower recall (0.76) than "Not Survived" (0.87) — meaning the model misses more actual survivors than non-survivors, something accuracy alone wouldn't reveal.

### 7. Hyperparameter Tuning (GridSearchCV)

In [24]:
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'solver': ['liblinear', 'lbfgs']
}

grid_search = GridSearchCV(LogisticRegression(max_iter=1000), param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best Cross-Validation Accuracy:", grid_search.best_score_)

Best Parameters: {'C': 1, 'solver': 'liblinear'}
Best Cross-Validation Accuracy: 0.7948882103811681


### 8. Evaluate Tuned Model

In [25]:
# Evaluate the tuned model on the test set
best_model = grid_search.best_estimator_
y_pred_tuned = best_model.predict(X_test)

print("Tuned Model Accuracy:", accuracy_score(y_test, y_pred_tuned))
print("\nTuned Classification Report:")
print(classification_report(y_test, y_pred_tuned, target_names=['Not Survived', 'Survived']))

Tuned Model Accuracy: 0.8044692737430168

Tuned Classification Report:
              precision    recall  f1-score   support

Not Survived       0.82      0.86      0.84       105
    Survived       0.78      0.73      0.76        74

    accuracy                           0.80       179
   macro avg       0.80      0.79      0.80       179
weighted avg       0.80      0.80      0.80       179



### 9. Before vs After Tuning

| Metric | Original Model | Tuned Model |
|---|---|---|
| Accuracy | 82.1% | 80.4% |
| Precision (Survived) | 0.80 | 0.78 |
| Recall (Survived) | 0.76 | 0.73 |
| F1-score (Survived) | 0.78 | 0.76 |
| Best Parameters | Default (`C=1.0`, `solver='lbfgs'`) | `C=1`, `solver='liblinear'` |

**Observation:** Hyperparameter tuning did **not** improve performance on this 
test set — the tuned model actually scored slightly lower across all metrics. 
This is a useful reminder that GridSearchCV optimizes for cross-validation 
performance, which doesn't always translate to a better result on one 
specific train-test split. With a small dataset like this (891 rows), such 
small differences can also just be noise rather than a real effect.